## Improve API-derived data catalog JSON

This notebook loads the JSON exports created by `api_batch_generator.ipynb`, aggregates dataset-level `infectiousAgent`, `species`, `topicCategory`, and `measurementTechnique` objects by source from `https://api.data.niaid.nih.gov/v1/query`, de-duplicates them per source, and writes the enriched JSON back to `../nde-metadata-corrections/metadata_for_DDE/dataCatalogs/`.


In [ ]:
import json
from datetime import datetime
from pathlib import Path

import requests


In [ ]:
script_path = Path.cwd()
parent_path = script_path.parent
result_path = parent_path / 'nde-metadata-corrections' / 'metadata_for_DDE' / 'dataCatalogs'
query_url = 'https://api.data.niaid.nih.gov/v1/query'
request_timeout = 60

print(f'Using data catalog directory: {result_path}')


In [ ]:
def load_latest_batch_file(directory: Path) -> tuple[Path, list[dict]]:
    batch_files = sorted(directory.glob('*_api_batch_file.json'))
    if not batch_files:
        raise FileNotFoundError(f'No API batch files found in {directory}')
    batch_path = batch_files[-1]
    return batch_path, json.loads(batch_path.read_text(encoding='utf-8'))


def safe_filename(value: str) -> str:
    return ''.join('_' if char in '\\/:*?"<>|' else char for char in value).strip() or 'data_catalog'


def build_record_filename(record: dict) -> str:
    filename_root = record.get('identifier') or record.get('name', 'data_catalog')
    return f'{safe_filename(filename_root)}.json'


def load_exported_records(directory: Path) -> tuple[Path, list[dict], dict[str, dict]]:
    batch_path, batch_records = load_latest_batch_file(directory)
    record_file_map = {}
    for record in batch_records:
        filename = build_record_filename(record)
        record_path = directory / filename
        if record_path.exists():
            record_file_map[filename] = json.loads(record_path.read_text(encoding='utf-8'))
        else:
            record_file_map[filename] = record.copy()
    return batch_path, batch_records, record_file_map


def fetch_query_page(params: dict | None = None, scroll_id: str | None = None) -> dict:
    request_params = {'scroll_id': scroll_id} if scroll_id else params
    response = requests.get(query_url, params=request_params, timeout=request_timeout)
    response.raise_for_status()
    return response.json()


def clean_nones(value):
    if isinstance(value, dict):
        cleaned = {}
        for key, item in value.items():
            if item in [None, '', [], {}, 'None']:
                continue
            cleaned[key] = clean_nones(item)
        return cleaned
    if isinstance(value, list):
        cleaned = [clean_nones(item) for item in value]
        return [item for item in cleaned if item not in [None, '', [], {}, 'None']]
    return value


def normalize_to_list(value):
    if isinstance(value, list):
        return value
    if value in [None, '', [], {}, 'None']:
        return []
    return [value]


In [ ]:
def aggregate_property_by_source(source_names: list[str], property_name: str) -> dict[str, list[dict]]:
    source_name_set = {name for name in source_names if name}
    aggregated = {name: {} for name in source_name_set}
    payload = fetch_query_page(
        params={
            'q': f'_exists_:{property_name}',
            'fields': f'{property_name},includedInDataCatalog.name',
            'size': 1000,
            'fetch_all': 'true'
        }
    )

    batch_count = 0
    hit_count = 0
    while True:
        hits = payload.get('hits', [])
        if not hits:
            break

        batch_count += 1
        hit_count += len(hits)
        for hit in hits:
            catalog_names = {
                entry.get('name')
                for entry in normalize_to_list(hit.get('includedInDataCatalog'))
                if isinstance(entry, dict) and entry.get('name')
            }
            matched_sources = source_name_set.intersection(catalog_names)
            if not matched_sources:
                continue

            for prop_value in normalize_to_list(hit.get(property_name)):
                if not isinstance(prop_value, dict):
                    continue
                clean_value = clean_nones(prop_value)
                dedupe_key = json.dumps(clean_value, sort_keys=True, ensure_ascii=False)
                for source_name in matched_sources:
                    aggregated[source_name][dedupe_key] = clean_value

        if batch_count % 25 == 0:
            print(f'Processed {hit_count} records with {property_name} across {batch_count} batches')

        scroll_id = payload.get('_scroll_id')
        if not scroll_id:
            break
        payload = fetch_query_page(scroll_id=scroll_id)

    print(f'Processed {hit_count} records with {property_name} in total')
    return {
        source_name: list(property_map.values())
        for source_name, property_map in aggregated.items()
        if property_map
    }


def aggregate_properties_by_source(source_names: list[str], property_names: list[str]) -> dict[str, dict[str, list[dict]]]:
    aggregated = {}
    for property_name in property_names:
        aggregated[property_name] = aggregate_property_by_source(source_names, property_name)
    return aggregated


In [ ]:
batch_path, batchlist, exported_record_map = load_exported_records(result_path)
source_names = [record.get('name') for record in batchlist]
properties_to_aggregate = ['infectiousAgent', 'species', 'topicCategory', 'measurementTechnique']
aggregated_property_map = aggregate_properties_by_source(source_names, properties_to_aggregate)

for record in batchlist:
    for property_name in properties_to_aggregate:
        values = aggregated_property_map.get(property_name, {}).get(record.get('name'))
        if values:
            record[property_name] = values
        elif property_name in record:
            del record[property_name]

for filename, record in exported_record_map.items():
    for property_name in properties_to_aggregate:
        values = aggregated_property_map.get(property_name, {}).get(record.get('name'))
        if values:
            record[property_name] = values
        elif property_name in record:
            del record[property_name]

print(f'Loaded batch file: {batch_path.name}')
print(f'Loaded {len(exported_record_map)} individual exported JSON files')
for property_name in properties_to_aggregate:
    print(f'Records with {property_name}: {sum(1 for record in batchlist if property_name in record)}')
print(batchlist[0])


In [ ]:
today = datetime.now().strftime('%Y-%m-%d')
batch_filepath = result_path / f'{today}_api_batch_file.json'

with batch_filepath.open('w', encoding='utf-8') as outfile:
    json.dump(batchlist, outfile, indent=4, ensure_ascii=False)

for filename, record in exported_record_map.items():
    output_filepath = result_path / filename
    with output_filepath.open('w', encoding='utf-8') as outfile:
        json.dump(record, outfile, indent=4, ensure_ascii=False)

print(f'Wrote enriched batch file: {batch_filepath}')
print(f'Updated {len(batchlist)} individual data catalog files')


In [ ]:
summary = {
    'records': len(batchlist),
    'records_with_infectiousAgent': sum(1 for record in batchlist if 'infectiousAgent' in record),
    'records_with_species': sum(1 for record in batchlist if 'species' in record),
    'records_with_topicCategory': sum(1 for record in batchlist if 'topicCategory' in record),
    'records_with_measurementTechnique': sum(1 for record in batchlist if 'measurementTechnique' in record),
    'total_unique_infectiousAgent_objects': sum(len(record.get('infectiousAgent', [])) for record in batchlist),
    'total_unique_species_objects': sum(len(record.get('species', [])) for record in batchlist),
    'total_unique_topicCategory_objects': sum(len(record.get('topicCategory', [])) for record in batchlist),
    'total_unique_measurementTechnique_objects': sum(len(record.get('measurementTechnique', [])) for record in batchlist)
}
summary
